# 공연 시간대 확장 전체 실행

기존 00_전체실행.ipynb의 최종 데이터가 있는 상태에서 실행한다. 09→10→11을 각각 새 커널에서 실행하고 결과와 읽기용 HTML을 저장한다. **기존 01~08, 원자료, 이전 통계 결과는 수정하지 않는다.**

- 09: 요일별 예정 일정·러닝타임 → 시간창 노출 및 가용률
- 10: 원자료의 여섯 시간 구간 매출 → 결측을 보존한 패널
- 11: 사전 고정한 계획에 따른 공통·업종별 기울기, 민감도, 해석

예정 종료 후 0~3시간을 주분석, 3~6시간·누적 0~6시간·시작 전 3시간을 보조로 본다. 정확한 공연일별 사후 매출을 복원하는 분석은 아니다. 네트워크 요청과 신규 수집은 없다.

이 PC의 `졸업프로젝트 (Python)` 또는 C:/Users/mamap/.cache/grad-notebook-venv/Scripts/python.exe 환경을 사용한다. 중복 실행하지 않는다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, time
import nbformat
from nbclient import NotebookClient
from nbconvert import HTMLExporter
from IPython.display import display, Markdown
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'data/raw').is_dir())
OUT=ROOT/'results/main/time_windows'
READABLE=ROOT/'reports/main/time_windows'
HTML=READABLE/'notebooks'; HTML.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
for name in ['OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS']: os.environ[name]='1'
NAMES=['09_공연시간대_노출구축.ipynb','10_시간대매출_패널.ipynb','11_공연시간창_매출분석.ipynb']
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
manifest={'started_utc':datetime.now(timezone.utc).isoformat(),'status':'running','steps':[],'network_requests':0}
manifest_path=OUT/'execution_manifest.json'
def save_manifest(): manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
save_manifest()

## 세 노트북을 새 커널로 실행

각 단계가 실패하면 오류가 있는 실행 노트북을 별도 failed 폴더에 보관하고 중단한다. 입력·결과 해시 검증도 각 단계 안에서 수행한다.

In [2]:
for name in NAMES:
    path=ROOT/'notebooks/main/analysis'/name
    notebook=nbformat.read(path,as_version=4)
    source=chr(10).join(c.source for c in notebook.cells)
    started=time.perf_counter()
    print('실행:',name,flush=True)
    try:
        NotebookClient(notebook,timeout=900,kernel_name='python3',resources={'metadata':{'path':str(path.parent)}}).execute()
        nbformat.validate(notebook); nbformat.write(notebook,path)
        exporter=HTMLExporter(template_name='lab',exclude_input=True,exclude_input_prompt=True,exclude_output_prompt=True)
        body,_=exporter.from_notebook_node(notebook)
        html_path=HTML/(path.stem+'.html'); html_path.write_text(body,encoding='utf-8')
        if name==NAMES[-1]: (READABLE/'공연시간창_분석보고서.html').write_text(body,encoding='utf-8')
        manifest['steps'].append({'notebook':path.relative_to(ROOT).as_posix(),'status':'passed',
          'seconds':round(time.perf_counter()-started,2),'code_cells':sum(c.cell_type=='code' for c in notebook.cells),
          'source_sha256':hashlib.sha256(source.encode('utf-8')).hexdigest(),'executed_sha256':sha(path),
          'html':html_path.relative_to(ROOT).as_posix()})
        save_manifest()
    except Exception as exc:
        failed=OUT/'failed'; failed.mkdir(exist_ok=True); nbformat.write(notebook,failed/name)
        manifest['status']='failed'; manifest['failed_notebook']=name
        manifest['error_type']=type(exc).__name__; save_manifest(); raise
manifest['status']='completed'; manifest['finished_utc']=datetime.now(timezone.utc).isoformat(); save_manifest()
display(pd.DataFrame(manifest['steps'])[['notebook','status','seconds','code_cells']])

실행: 09_공연시간대_노출구축.ipynb


C:\Users\mamap\.cache\grad-notebook-venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()


실행: 10_시간대매출_패널.ipynb


실행: 11_공연시간창_매출분석.ipynb


,notebook,status,seconds,code_cells
0,notebooks/main/analysis/09_공연시간대_노출구축.ipynb,passed,37.76,17
1,notebooks/main/analysis/10_시간대매출_패널.ipynb,passed,3.99,7
2,notebooks/main/analysis/11_공연시간창_매출분석.ipynb,passed,6.70,14


## 결과 읽기

- 최종 보고서: `reports/main/time_windows/공연시간창_분석보고서.html`
- 주모형·업종별 결과: `11_common_effects.csv`, `11_industry_effects.csv`
- 민감도와 검증: `11_sensitivity.csv`, `11_implementation_checks.csv`
- 실행 기록: `results/main/time_windows/execution_manifest.json`

HTML은 계산을 포함한 노트북의 출력과 해석을 모은 읽기용 파일이다. 재현할 때는 이 실행 노트북을 사용한다.